In [10]:
from preprocessing.data_preprocessing import get_processed_data
from preprocessing.utils import clean_chunk

PDF_PATH = '../data/coffee_processing.pdf'

In [11]:
final_data = get_processed_data(PDF_PATH)
print(final_data)

Total blocks: 21 | {'text': 17, 'table': 2, 'image': 2}
  [table] p2 → summarized & stored
  [image] p2 → summarized & stored
  [image] p2 → summarized & stored
  [table] p4 → summarized & stored
{'texts': ['A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplis

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " "]
)
text_chunks = text_splitter.create_documents(final_data['texts'])


cleaned_documents = []

for i, chunk in enumerate(text_chunks):
    raw_text = chunk.page_content
    cleaned_text = clean_chunk(raw_text)
    
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)

for i, text_summary in enumerate(final_data['images']):
    cleaned_text = clean_chunk(text_summary)
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)

for i, text_summary in enumerate(final_data['tables']):
    cleaned_text = clean_chunk(text_summary)
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)


for doc in cleaned_documents:
    print(f"{'-' * 40} {len(doc.page_content)} {'-' * 40}")
    print(doc.page_content)


---------------------------------------- 415 ----------------------------------------
A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices
---------------------------------------- 456 ----------------------------------------
The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-d

In [18]:
import os
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from dotenv import load_dotenv

load_dotenv('/Users/nnandagopal/Desktop/personal_projects/RAG/.env')


# ── Index must use dotproduct for hybrid search ───────────────────────────────
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index = 'rag-pipeline'

pc.create_index(
    name=index,
    dimension=1536,
    metric="dotproduct",          # required for sparse-dense hybrid
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

index = pc.Index(index)

In [19]:
def fit_bm25_encoder(cleaned_documents: list) -> BM25Encoder:
    """
    Fit Pinecone's BM25Encoder on your corpus.
    This learns term frequencies across all documents —
    same math as classic BM25 but outputs sparse vectors
    that Pinecone can store and query natively.

    Must be fit before encoding any documents or queries.
    Save it after fitting — refitting is expensive on large corpus.
    """
    all_texts = [
        chunk["page_content"] if isinstance(chunk, dict) else chunk.page_content
        for chunk in cleaned_documents
    ]

    bm25 = BM25Encoder()
    bm25.fit(all_texts)          # learns IDF weights from corpus
    bm25.dump("bm25_encoder.json")  # save so you don't refit every run
    print("BM25 encoder fitted and saved.")
    return bm25


# Load if already fitted:
# bm25 = BM25Encoder().load("bm25_encoder.json")

bm25_encoder = fit_bm25_encoder(cleaned_documents)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nnandagopal/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nnandagopal/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
100%|██████████| 16/16 [00:00<00:00, 504.00it/s]

BM25 encoder fitted and saved.


In [20]:
def upsert_hybrid(index, embeddings_model, bm25_encoder, cleaned_documents, batch_size=100):
    """
    Upsert both dense (OpenAI) and sparse (BM25) vectors per document.
    Pinecone stores them together — one upsert call, one document entry.

    Each vector in Pinecone now has:
        values        → dense embedding (semantic)
        sparse_values → BM25 sparse vector (keyword)
    """
    all_texts, all_metadatas = [], []
    for chunk in cleaned_documents:
        text     = chunk["page_content"] if isinstance(chunk, dict) else chunk.page_content
        metadata = chunk.get("metadata", {}) if isinstance(chunk, dict) else chunk.metadata
        all_texts.append(text)
        all_metadatas.append(metadata)

    total_batches = (len(all_texts) + batch_size - 1) // batch_size

    for i in range(0, len(all_texts), batch_size):
        batch_texts     = all_texts[i : i + batch_size]
        batch_metadatas = all_metadatas[i : i + batch_size]

        dense_embeddings  = embeddings_model.embed_documents(batch_texts)
        sparse_embeddings = bm25_encoder.encode_documents(batch_texts)  # BM25 sparse vectors

        vectors = [
            {
                "id":            f"doc_{i + j}",
                "values":        dense_embeddings[j],          # dense
                "sparse_values": sparse_embeddings[j],         # sparse BM25
                "metadata":      {**batch_metadatas[j], "text": batch_texts[j]}
            }
            for j in range(len(batch_texts))
        ]

        index.upsert(vectors=vectors)
        print(f"  Batch {(i // batch_size)+1}/{total_batches} upserted")


upsert_hybrid(index, embeddings_model, bm25_encoder, cleaned_documents)

  Batch 1/1 upserted


In [48]:
def retrieve_hybrid_pinecone(index, embeddings_model, bm25_encoder,
                              query: str, top_k: int = 10, alpha: float = 0.5) -> list[dict]:
    """
    Query Pinecone with both dense and sparse vectors simultaneously.

    alpha controls the dense vs sparse balance:
        alpha = 1.0  → pure vector search (semantic only)
        alpha = 0.0  → pure BM25 (keyword only)
        alpha = 0.5  → equal blend (good default)

    Pinecone handles the fusion internally — you get one ranked result list.
    The score returned is a dotproduct combination of both signals.

    Args:
        alpha: Weight for dense vs sparse (0.0 to 1.0)
    """
    dense_vector  = embeddings_model.embed_query(query)
    sparse_vector = bm25_encoder.encode_queries(query)   # BM25 sparse for query

    # Scale vectors by alpha to control blend
    scaled_dense  = [v * alpha for v in dense_vector]
    scaled_sparse = {
        "indices": sparse_vector["indices"],
        "values":  [v * (1 - alpha) for v in sparse_vector["values"]]
    }

    results = index.query(
        vector=scaled_dense,
        sparse_vector=scaled_sparse,
        top_k=top_k,
        include_metadata=True
    )

    return [
        {
            "text":         match["metadata"].get("text", ""),
            "metadata":     {k: v for k, v in match["metadata"].items() if k != "text"},  # exclude text from metadata
            "hybrid_score": match["score"],
            "cohere_score":  None,
            "context_score": None
        }
        for match in results["matches"]
    ]

# query = """What are the environmental concerns and solutions related to coffee processing"""
query = '''how choice of processing method can enhance certain flavor characteristics of coffe'''

hybrid_results = retrieve_hybrid_pinecone(
    index, embeddings_model, bm25_encoder,
    query=query, top_k=10, alpha=0.5
)

In [49]:
for i in hybrid_results:
    print(i, end='\n\n')

{'text': 'The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical processes', 'metadata': {'chunk_id': 1, 'source': 'coffee_processing_guide'}, 'hybrid_score': 0.585569263, 'cohere_score': None, 'context_score': None}

{'text': 'A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available r

In [50]:
import re

def contextual_matching(query: str, docs: list[dict]) -> list[dict]:
    """
    Score each document by simple term overlap with the query.
    Pure Python — no API calls, no models.

    This catches cases where an exact query word appears in the document
    but wasn't strongly weighted by BM25 or vector search.

    Context score meaning:
        1.0 → every query word appears in the document
        0.5 → half the query words appear
        0.0 → no query words found in document

    Args:
        query: Search query string
        docs: Merged document list

    Returns:
        Same docs with context_score populated
    """
    query_terms = set(re.sub(r'[^\w\s]', '', query.lower()).split())

    for doc in docs:
        doc_terms = set(re.sub(r'[^\w\s]', '', doc["text"].lower()).split())
        overlap = len(query_terms & doc_terms) / max(len(query_terms), 1)
        doc["context_score"] = overlap

    return docs


merged_docs = contextual_matching(query, hybrid_results)

merged_docs

[{'text': 'The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical processes',
  'metadata': {'chunk_id': 1, 'source': 'coffee_processing_guide'},
  'hybrid_score': 0.585569263,
  'cohere_score': None,
  'context_score': 0.8181818181818182},
 {'text': 'A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on th

In [51]:
import cohere

def rerank_with_cohere(query: str, docs: list[dict], api_key: str) -> list[dict]:
    """
    Rerank all merged documents using Cohere's reranker.

    Unlike embedding similarity (which compares vectors independently),
    Cohere's reranker reads the query AND document together — much more accurate.

    Cohere score meaning:
        Close to 1.0 → highly relevant to the query
        Close to 0.0 → not relevant
        This is a relevance probability, not a distance.

    Args:
        query: Search query string
        docs: Merged document list
        api_key: Cohere API key (get from dashboard.cohere.com)

    Returns:
        Same docs with cohere_score populated (order unchanged — fusion handles ranking)
    """
    co    = cohere.Client(api_key)
    texts = [doc["text"] for doc in docs]

    response = co.rerank(
        model="rerank-english-v3.0",
        query=query,
        documents=texts,
        top_n=len(texts)   # score all, let fusion decide final ranking
    )

    for result in response.results:
        docs[result.index]["cohere_score"] = result.relevance_score

    return docs


merged_docs = rerank_with_cohere(query, merged_docs, api_key=os.getenv('COHERE_API_KEY'))
merged_docs

[{'text': 'The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical processes',
  'metadata': {'chunk_id': 1, 'source': 'coffee_processing_guide'},
  'hybrid_score': 0.585569263,
  'cohere_score': 0.99986553,
  'context_score': 0.8181818181818182},
 {'text': 'A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based

In [52]:
def normalize_scores(docs: list[dict], score_keys: list[str]) -> list[dict]:
    """
    Min-max normalize each score across all documents.

    Why even when scores are 0-1:
        Scores may all be valid 0-1 but clustered in a tiny range.
        e.g. hybrid scores [0.30, 0.31, 0.32] vs cohere [0.01, 0.50, 0.99]
        Without normalization, cohere dominates the weighted sum purely
        due to spread — not because of its weight. Normalization ensures
        each method's relative ranking contributes fairly.

    Args:
        docs: Document list with raw scores
        score_keys: Which score fields to normalize

    Returns:
        Same docs with scores normalized in-place
    """
    for key in score_keys:
        scores = [doc.get(key, 0) for doc in docs]
        min_s  = min(scores)
        max_s  = max(scores)
        spread = max_s - min_s

        for doc in docs:
            raw = doc.get(key, 0)
            if spread > 1e-6:
                doc[key] = (raw - min_s) / spread   # normal case
            else:
                doc[key] = 1.0 # all scores identical → all equally relevant

    return docs


def fuse_and_rank(docs: list[dict], weights: dict, top_k: int = 5) -> list[dict]:
    """
    Combine all normalized scores into a single final score and rank.

    Weights control how much each signal matters:
        hybrid_score  → semantic similarity & keyword exact match(main signal)
        cohere_score  → deep contextual relevance (most accurate)
        context_score → simple term overlap (lightweight boost)

    Final score = weighted sum of all normalized scores (0 to 1).

    Args:
        docs: Document list with normalized scores
        weights: Dict of {score_key: weight} — should sum to 1.0
        top_k: How many final results to return

    Returns:
        Top-k documents sorted by final_score descending
    """
    for doc in docs:
        doc["final_score"] = sum(
            weights.get(key, 0) * doc.get(key, 0)
            for key in weights
        )

    return sorted(docs, key=lambda x: x["final_score"], reverse=True)[:top_k]


SCORE_KEYS = ["hybrid_score", "cohere_score", "context_score"]

WEIGHTS = {
    "hybrid_score":  0.40,   # Pinecone already blended dense + BM25
    "cohere_score":  0.50,   # reranker — most accurate signal
    "context_score": 0.10    # lightweight term overlap boost
}

merged_docs = normalize_scores(merged_docs, SCORE_KEYS)
final_results = fuse_and_rank(merged_docs, WEIGHTS, top_k=5)
final_results

[{'text': 'The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical processes',
  'metadata': {'chunk_id': 1, 'source': 'coffee_processing_guide'},
  'hybrid_score': 1.0,
  'cohere_score': 1.0,
  'context_score': 1.0,
  'final_score': 1.0},
 {'text': 'A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on thei